In [1]:
import pandas as pd
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler,OneHotEncoder
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

warnings.filterwarnings('ignore')

In [2]:
churn= pd.read_csv(r'C:\Users\suraj\Downloads\Bank_Customer_Churn.csv')
churn

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,15760299,717,Spain,Male,28,7,89810.96,2,0,1,132588.21,0
9996,15644026,707,Germany,Female,61,2,0.00,1,0,1,170404.98,0
9997,15790865,581,Germany,Male,30,0,0.00,1,1,1,171751.36,0
9998,15658563,676,France,Female,35,4,160742.97,1,1,1,25963.59,1


In [7]:
not_important= 'customer_id'
target_field= 'churn'
fields= list(churn.columns)
fields.remove(target_field)
fields.remove(not_important)
input_fields= fields
input_fields


['credit_score',
 'country',
 'gender',
 'age',
 'tenure',
 'balance',
 'products_number',
 'credit_card',
 'active_member',
 'estimated_salary']

In [8]:
category_fields= list(churn[input_fields].select_dtypes(exclude='number').columns)
numeric_fields= list(churn[input_fields].select_dtypes(include='number').columns)


In [9]:
training_data,test_data= train_test_split(churn,test_size=0.3,random_state=4)

In [10]:
scaler= RobustScaler().fit(training_data[numeric_fields])
training_data.loc[:,numeric_fields]= scaler.transform(training_data[numeric_fields])
test_data.loc[:,numeric_fields]= scaler.transform(test_data[numeric_fields])

In [11]:
encoder= OneHotEncoder().fit(training_data[category_fields])
training_data.loc[:,encoder.get_feature_names_out()]= encoder.transform(training_data[category_fields]).toarray()
test_data.loc[:,encoder.get_feature_names_out()]= encoder.transform(test_data[category_fields]).toarray()

In [12]:
input_fields= list(encoder.get_feature_names_out())+numeric_fields

In [13]:
model= LogisticRegression().fit(training_data[input_fields],training_data[target_field])
predictions=model.predict(test_data[input_fields])

In [14]:
print(f'accuracy_score:{accuracy_score(test_data[target_field], predictions)}')
print(f'precision_score:{precision_score(test_data[target_field], predictions)}')
print(f'sensivity_score:{recall_score(test_data[target_field], predictions)}')
print(f'f1_score:{f1_score(test_data[target_field], predictions)}')

accuracy_score:0.7733333333333333
precision_score:0.5882352941176471
sensivity_score:0.21978021978021978
f1_score:0.32


In [15]:
model.coef_

array([[-0.65035497,  0.32899626, -0.66555117, -0.50037226, -0.48653762,
         0.05655411,  0.85906877, -0.07330398,  0.55018539,  0.02430909,
        -0.00207254, -0.91146521, -0.01281783]])

In [16]:
weights=pd.DataFrame({'features': model.feature_names_in_.reshape(-1),'weight': model.coef_.reshape(-1)})
weights.sort_values(by=['weight'],ascending=False)

,features,weight
6,age,0.859069
8,balance,0.550185
1,country_Germany,0.328996
5,credit_score,0.056554
9,products_number,0.024309
10,credit_card,-0.002073
12,estimated_salary,-0.012818
7,tenure,-0.073304
4,gender_Male,-0.486538
3,gender_Female,-0.500372


In [17]:
traget_field='churn'
input_fields= ['age','balance','country','gender','active_member']

category_fields= list(churn[input_fields].select_dtypes(exclude='number').columns)
numeric_fields= list(churn[input_fields].select_dtypes(include='number').columns)

training_data,test_data= train_test_split(churn,test_size=0.3,random_state=4)

scaler= RobustScaler().fit(training_data[numeric_fields])
training_data.loc[:,numeric_fields]= scaler.transform(training_data[numeric_fields])
test_data.loc[:,numeric_fields]= scaler.transform(test_data[numeric_fields])

encoder= OneHotEncoder().fit(training_data[category_fields])
training_data.loc[:,encoder.get_feature_names_out()]= encoder.transform(training_data[category_fields]).toarray()
test_data.loc[:,encoder.get_feature_names_out()]= encoder.transform(test_data[category_fields]).toarray()


model_input_fields= list(encoder.get_feature_names_out())+numeric_fields

model= LogisticRegression().fit(training_data[model_input_fields],training_data[target_field])
predictions=model.predict(test_data[model_input_fields])

In [18]:
print(f'accuracy: {accuracy_score(test_data[target_field], predictions)}')
print(f'precision: {precision_score(test_data[target_field], predictions)}')
print(f'recall: {recall_score(test_data[target_field], predictions)}')
print(f'f1: {f1_score(test_data[target_field], predictions)}')

accuracy: 0.7753333333333333
precision: 0.5978260869565217
recall: 0.22664835164835165
f1: 0.3286852589641434


In [19]:
import joblib as jb
churn_model= {'target_field': target_field
               ,'input_fields': input_fields
                ,'categorical_fields': category_fields
                 ,'numeric_fields': numeric_fields
                  ,'model_inputs': model_input_fields
                   ,'scaler': scaler
                    ,'encoder': encoder
                    ,'model': model}
jb.dump(churn_model, 'churn_model.joblib')

['churn_model.joblib']

In [ ]:
import joblib as jb
import pandas as pd

saved_model= jb.load('churn_model.joblib')
input_values={}
for feature in saved_model['input_fields']:
    value= input(f'enter the {feature.lower()}:')
    input_values.update({feature:value})

for field in saved_model['numeric_fields']:
    input_values[field]= float(input_values[field])
    
input_record= pd.DataFrame(input_values, index=[0])

input_record[saved_model['numeric_fields']]= saved_model['scaler'].transform(input_record[saved_model['numeric_fields']])

input_record.loc[:,saved_model['encoder'].get_feature_names_out()]= saved_model['encoder'].transform(input_record[saved_model['categorical_fields']]).toarray()

if saved_model['model'].predict(input_record[saved_model['model_inputs']]) == [1]:
    prediction= 'churn'
else:
    prediction= 'wont_churn'
print(f'pridicted_status:{prediction}')



In [16]:
churn_model['numeric_fields']

['age', 'balance', 'active_member']

In [25]:
for feature in saved_model['input_fields']:
    value= input(f'enter the {feature.lower()}:')
    input_values.update({feature:value})

enter the age: 34
enter the balance: 34
enter the country: 34
enter the gender: 2
enter the active_member: 


In [25]:
input_record

,age,balance,country,gender,active_member,country_France,country_Germany,country_Spain,gender_Female,gender_Male
0,0.727273,-0.457678,Germany,Female,5.0,0.0,1.0,0.0,1.0,0.0
